# 10. Lecke - AI ügynökök éles környezetben

Ebben a leckében megismered az AI ügynökök **éles környezetben alkalmazható mintáit** a Microsoft Agent Framework (Python) segítségével. Témáink:

- **Megfigyelhetőség** — időzítés és naplózás hozzáadása az ügynökök interakcióihoz
- **Értékelés** — egy értékelő ügynök használata a válaszminőség pontozására
- **Költségkezelés** — stratégiák token optimalizálásra és modell kiválasztásra

A forgatókönyv egy **utazási ügynök**, amely segít a felhasználóknak utazások tervezésében, felügyelettel és értékeléssel kiegészítve.


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Gyártási szempontok

Az AI ágensek prototípusból termelésbe vitele három pillér alapos figyelmét igényli:

1. **Megfigyelhetőség** — Látnod kell, hogy az ügynök mit csinál, mennyi ideig tart, és mely eszközöket hívja meg. Nyomkövetés és naplózás nélkül szinte lehetetlen a gyártási problémák hibakeresése.

2. **Értékelés** — Az automatizált minőségellenőrzés biztosítja, hogy az ügynök válaszai idővel pontosak, teljesek és segítőkészek maradjanak. Egy értékelő ügynök megadhat pontszámot a válaszoknak a meghatározott kritériumok alapján.

3. **Költségmenedzsment** — A tokenhasználat közvetlenül befolyásolja a költségeket. Olyan stratégiák, mint az utasítás optimalizálása, modellválasztás és gyorsítótárazás, segítenek a kiadások kordában tartásában a minőség feláldozása nélkül.


## Egy megfigyelhető ügynök létrehozása

Meghatározzuk az utazási eszközöket, és időzítéssel vesszük körbe az ügynök hívását, hogy figyelemmel kísérhessük a késleltetést. Éles környezetben az OpenTelemetry-vel vagy hasonló trace-elési háttérrendszerrel integrálnád.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Értékelési minták

Egy gyakori gyártási mintázat, hogy egy második ügynököt használnak **értékelőként**. Az értékelő pontozza a fő ügynök válaszát előre meghatározott szempontok alapján, mint például teljesség, pontosság és hasznosság.

Ez lehetővé teszi:
- Automatikus minőségi kapuk bevezetését, mielőtt a válaszok elérnék a felhasználókat
- Regresszió felismerését, amikor a bemenetek vagy a modellek változnak
- Az ügynök teljesítményének folyamatos nyomon követését az idő során


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Költségkezelési stratégiák

A költségek ellenőrzése kritikus a termelő AI-ügynökök számára. Íme a kulcsfontosságú stratégiák:

| Stratégia | Leírás |
|---|---|
| **Parancsoptimalizálás** | Tartsa tömören a rendszerutasításokat. Távolítsa el a redundáns kontextust az input tokenek csökkentése érdekében. |
    "| **Modell kiválasztása** | Használjon kisebb, olcsóbb modelleket (pl. GPT-5-mini) egyszerű feladatokhoz, mint például osztályozás vagy kivonás, és tartsa fenn a nagyobb modelleket összetettebb érvelésekhez. |\n",
| **Gyorsítótárazás** | Gyorsítótárazza az eszköz eredményeket és a gyakori lekérdezéseket, hogy elkerülje az ismétlődő API-hívásokat. |
| **Token költségkeret** | Állítson be `max_tokens` korlátokat az előre nem látható hosszú válaszok megakadályozására. |
| **Csoportosítás** | Csoportosítson több felhasználói lekérdezést egyetlen API-hívásba, ahol lehetséges. |

A gyakorlatban jól működik egy többszintű megközelítés: egyszerű kéréseket egy gyors, olcsó modellhez irányít, míg csak az összetett lekérdezéseket bonyolultabbhoz továbbítja.


## Összefoglaló

Ebben a leckében megtanultad, hogyan:

1. **Adj megfigyelhetőséget** az ügynökök interakcióihoz időzítéssel és naplózással, megteremtve az alapot a követéshez és monitorozáshoz.
2. **Értékeld automatikusan az ügynök válaszait** egy értékelő ügynök segítségével, amely pontozza a teljességet, pontosságot és hasznosságot.
3. **Kezeld a költségeket** prompt optimalizálással, modellválasztással, gyorsítótárazással és token költségvetéssel.

Ezek a gyártási minták segítenek biztosítani, hogy az AI ügynökeid megbízhatóak, mérhetőek és költséghatékonyak legyenek nagy léptékben.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
